# Glue-Style Jobs: Parameters, Metrics, Bookmarks, a Complete Job

## What you'll practice

AWS Glue jobs receive parameters as `--KEY value` arguments, read config from S3, track progress with **job bookmarks**, and emit logs/metrics to CloudWatch. Here you'll
build each of those pieces in plain Python against the local S3 folder, ending with a full job you can run from the terminal.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/aws/test_11_02_glue_style_jobs.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/11_python_for_aws/02_glue_style_jobs.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
S3_ROOT = LAB / "datasets/s3"        # local stand-in for S3: <root>/<bucket>/<key>
for p in sorted(S3_ROOT.rglob("*"))[:12]:
    print(p.relative_to(S3_ROOT))

## Exercise 1 — 🟢 Easy · getResolvedOptions

**Implement:** `get_resolved_options(argv: list[str], names: list[str]) -> dict[str, str]`

Mimic `awsglue.utils.getResolvedOptions(sys.argv, ["JOB_NAME", "input_path"])`: find `--NAME value` (or `--NAME=value`) pairs for the requested names, ignoring everything else
(Glue passes many internal args). Missing a requested name → `ValueError` listing all missing names. The last occurrence wins.

**Input:** `argv: list[str], names: list[str]`

**Expected output:** `dict[str, str]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_get_resolved_options` in `tests/aws/test_11_02_glue_style_jobs.py`.

**Check:** `pytest tests/aws/test_11_02_glue_style_jobs.py -k test_get_resolved_options`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · A proper CLI for local runs

**Implement:** `build_parser() -> argparse.ArgumentParser`

Build an `argparse` parser for the job: `--config` (required, an `s3://` URI — validate with a custom `type=` function), `--ingest-date` (optional, `YYYY-MM-DD`, validated, returned as `date`),
`--env` (choices dev/qa/prod, default dev), `--batch-size` (int ≥ 1, default 500), `--dry-run` (flag), `--log-level` (default INFO, case-insensitive → upper). Unknown arguments must be **tolerated**
by callers via `parse_known_args` (test does this). Bad values must make argparse exit with an error (raises `SystemExit`).

**Expected output:** `argparse.ArgumentParser`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_build_parser_ok, test_build_parser_errors` in `tests/aws/test_11_02_glue_style_jobs.py`.

**Check:** `pytest tests/aws/test_11_02_glue_style_jobs.py -k "test_build_parser_ok or test_build_parser_errors"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · CloudWatch Embedded Metric Format

**Implement:** `emf_record(namespace: str, metrics: dict[str, tuple[float, str]], dimensions: dict[str, str], timestamp_ms: int) -> dict`

Lambda/Glue logs printed in **EMF** JSON become CloudWatch metrics automatically. Build the record:
`{"_aws": {"Timestamp": ts, "CloudWatchMetrics": [{"Namespace": ns, "Dimensions": [[dim names...]], "Metrics": [{"Name": n, "Unit": u}, ...]}]}, <dim>: <value>, ..., <metric>: <value>, ...}`.
Validate: unit in `{"Count", "Seconds", "Milliseconds", "Bytes", "Percent", "None"}`; at most 30 dimensions; metric and dimension names must not collide → `ValueError`. Then log one per job run.

**Input:** `namespace: str, metrics: dict[str, tuple[float, str]], dimensions: dict[str, str], timestamp_ms: int`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_emf_record` in `tests/aws/test_11_02_glue_style_jobs.py`.

**Check:** `pytest tests/aws/test_11_02_glue_style_jobs.py -k test_emf_record`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Job bookmarks over S3 objects

**Implement:** `class JobBookmark`

Glue bookmarks remember which objects a job has processed. `JobBookmark(root, job_name)` stores state at `lab-processed-bucket/_bookmarks/<job_name>.json` in the local S3 root:
`{"objects": {"bucket/key": etag}, "runs": n}`. Methods: `filter_new(bucket, objects)` → objects (dicts with `Key`, `ETag`) that are new **or changed** (different ETag);
`mark(bucket, objects)` (staged in memory); `commit()` (persists staged marks atomically and increments `runs`); `reset()` (clears state on disk — like `--job-bookmark-option job-bookmark-reset`).
Uncommitted marks must not be visible to a new `JobBookmark` instance (so a failed run reprocesses).

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_job_bookmark` in `tests/aws/test_11_02_glue_style_jobs.py`.

**Check:** `pytest tests/aws/test_11_02_glue_style_jobs.py -k test_job_bookmark`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · A complete Glue-style job · 🧩 Mini-project

**Implement:** `run_sales_job(argv: list[str], root, now_ms: int | None = None) -> dict`

Assemble a job runnable as `python job.py --JOB_NAME daily_sales --config_uri s3://lab-config-bucket/jobs/daily_sales.json [--ingest_date 2026-01-10]` (argv parsing via your resolver; `ingest_date` optional).
1. Load the config JSON from local S3 (`source`, `target`, `rejected`, `partition_key`, `min_quantity`, `allowed_status`).
2. List source objects (`*.csv`) under `source`; group by partition (`ingest_date=...` path component). If `--ingest_date` is given, process only that partition (ignore the bookmark); otherwise
   process partitions containing objects the **bookmark** (`JobBookmark`-like, keyed by key+md5) says are new/changed.
3. Per partition: read all its CSVs, validate (quantity ≥ `min_quantity`, status in `allowed_status` case-insensitive, numeric price ≥ 0, order_id present), dedupe by `order_id` keeping latest
   `updated_at`, then **overwrite** `target/<partition_key>=<value>/part-00000.jsonl` and `rejected/<partition_key>=<value>/part-00000.jsonl`.
4. Commit the bookmark after all partitions succeed. Write a run summary to `target/_runs/<JOB_NAME>-<now_ms>.json` and include one EMF metric record in it under `"metrics"`.
Return the summary: `{"job", "partitions": {value: {"read", "valid", "rejected", "duplicates"}}, "status": "SUCCEEDED", "metrics": {...}}`. With nothing new, partitions is `{}`.

**Input:** `argv: list[str], root, now_ms: int | None = None`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_run_sales_job_bookmarked, test_run_sales_job_explicit_partition_and_args` in `tests/aws/test_11_02_glue_style_jobs.py`.

**Check:** `pytest tests/aws/test_11_02_glue_style_jobs.py -k "test_run_sales_job_bookmarked or test_run_sales_job_explicit_partition_and_args"`

**YOUR CODE ↓**